In [2]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install scikit-learn -q

In [4]:
import numpy as np, os, json
from sklearn.model_selection import train_test_split

LAND_DIR   = '/content/drive/MyDrive/Projets/sign-language-detection/01_Landmarks'
OUTPUT_DIR = '/content/drive/MyDrive/Projets/sign-language-detection/02_Dataset_Final'
CLASSES    = ['eat', 'drink', 'water', 'sleep', 'medicine']
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Chargement
X, y = [], []
for idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(LAND_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.endswith('.npy'):
            X.append(np.load(os.path.join(cls_dir, f)))
            y.append(idx)

X = np.array(X)
y = np.array(y)
print(f"Dataset : X={X.shape}, y={y.shape}")

# Split stratifié
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# Export
for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                  ('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    np.save(os.path.join(OUTPUT_DIR, f'{name}.npy'), arr)
    print(f"  Sauvegardé : {name}.npy {arr.shape}")

# Labels
with open(os.path.join(OUTPUT_DIR, 'labels.json'), 'w') as f:
    json.dump({cls: i for i, cls in enumerate(CLASSES)}, f, indent=2)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(" Export terminé")

Dataset : X=(45, 64, 1662), y=(45,)
  Sauvegardé : X_train.npy (31, 64, 1662)
  Sauvegardé : X_val.npy (7, 64, 1662)
  Sauvegardé : X_test.npy (7, 64, 1662)
  Sauvegardé : y_train.npy (31,)
  Sauvegardé : y_val.npy (7,)
  Sauvegardé : y_test.npy (7,)

Train: 31 | Val: 7 | Test: 7
 Export terminé


In [ ]:
print("=== VÉRIFICATION FINALE ===\n")

checks = {
    'X_train': (None, 64, 1662),
    'X_val'  : (None, 64, 1662),
    'X_test' : (None, 64, 1662),
}

tous_ok = True
for name, expected_suffix in checks.items():
    arr = np.load(os.path.join(OUTPUT_DIR, f'{name}.npy'))
    shape_ok = arr.shape[1:] == (64, 1662)
    nan_ok   = not np.isnan(arr).any()
    statut   = "Bon" if (shape_ok and nan_ok) else "Mauvaise"
    if not (shape_ok and nan_ok):
        tous_ok = False
    print(f"  {statut} {name}.npy — shape {arr.shape} | NaN: {np.isnan(arr).any()}")

print(f"\n  labels.json : {json.load(open(os.path.join(OUTPUT_DIR, 'labels.json')))}")

if tous_ok:
    print("\n Tous les fichiers sont propres — prêts pour le Membre 2")
else:
    print("\nProblème détecté — ne pas livrer avant correction")

=== VÉRIFICATION FINALE ===

  ✅ X_train.npy — shape (31, 64, 1662) | NaN: False
  ✅ X_val.npy — shape (7, 64, 1662) | NaN: False
  ✅ X_test.npy — shape (7, 64, 1662) | NaN: False

  labels.json : {'eat': 0, 'drink': 1, 'water': 2, 'sleep': 3, 'medicine': 4}

✅ Tous les fichiers sont propres — prêts pour le Membre 2
